In [1]:
import os
os.chdir(os.path.dirname(os.getcwd()))

In [2]:
from argparse import ArgumentParser
import numpy as np
import pandas as pd
import csv
import time
from scipy.stats import norm as norm_dist
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, train_test_split
from ngboost.distns import Bernoulli, Normal
from ngboost.scores import LogScore
from ngboost import NGBRegressor
from ngboost.learners import default_linear_learner, default_tree_learner

np.random.seed(1)

In [3]:
dataset_name_to_loader = {
    "Boston Housing": lambda: pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data",
        header=None,
        delim_whitespace=True,
    ),
    "Concrete Compression Strength": lambda: pd.read_excel(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
    ),
    "Energy Efficiency": lambda: pd.read_excel(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00242/ENB2012_data.xlsx"
    ).iloc[:, :-1],
    "Kin8nm": lambda: pd.read_csv("ngboost/data/uci/kin8nm.csv"),
    "Naval Propulsion": lambda: pd.read_csv(
        "ngboost/data/uci/naval-propulsion.txt", delim_whitespace=True, header=None
    ).iloc[:, :-1],
    "Combined Cycle Power Plant": lambda: pd.read_excel("ngboost/data/uci/power-plant.xlsx"),
    "Protein Structure": lambda: pd.read_csv("ngboost/data/uci/protein.csv")[
        ["F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8", "F9", "RMSD"]
    ],
    "Wine Quality Red": lambda: pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
        delimiter=";",
    ),
    "Yacht Hydrodynamics": lambda: pd.read_csv(
        "http://archive.ics.uci.edu/ml/machine-learning-databases/00243/yacht_hydrodynamics.data",
        header=None,
        delim_whitespace=True,
    ),
    "Year Prediciton MSD": lambda: pd.read_csv("ngboost/data/uci/YearPredictionMSD.txt").iloc[:, ::-1],
}

base_name_to_learner = {
    "tree": default_tree_learner,
    "linear": default_linear_learner,
}

dataset_list = ["Boston Housing", "Concrete Compression Strength", "Energy Efficiency", "Kin8nm", "Naval Propulsion", "Combined Cycle Power Plant", "Protein Structure", "Wine Quality Red", "Yacht Hydrodynamics", "Year Prediciton MSD"]

In [ ]:
args = {
    "dataset": "Concrete Compression Strength",
    "n_est": 2000,
    "n_splits": 20,
    "distn": "Normal",
    "lr": 0.01,
    "natural": "store_true",
    "score": "LogScore",
    "base": "tree",
    "minibatch_frac": 1.0,
    "verbose": True,
}
with open('ngboost_performance_logloss.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Dataset', 'NGBoost RMSE Mean', 'NGBoost RMSE Std', 'NGBoost NLL Mean', 'NGBoost NLL Std', 'Computation Time'])
    for dset in dataset_list:
        start_time = time.time()  # Start time measurement
        args["dataset"] = dset
        y_true, ngb_rmse, ngb_nll = [], [], []

        # Load dataset -- use last column as label
        data = dataset_name_to_loader[args['dataset']]()
        X, y = data.iloc[:, :-1].values, data.iloc[:, -1].values

        print(f"== Dataset={args['dataset']} X.shape={str(X.shape)} {args['score']}/{args['distn']}")

        lgbm_rmse = []
        if args["dataset"] == "Year Prediciton MSD":
            args["lr"] = 0.1
            folds = [(np.arange(463715), np.arange(463715, len(X)))]
            args["minibatch_frac"] = 0.1 
        elif args["dataset"] == "Protein Structure":
            args["lr"] = 0.01
            args["minibatch_frac"] = 1.0
            kf = KFold(n_splits=5)
            folds = kf.split(X)
            # Follow https://github.com/yaringal/DropoutUncertaintyExps/blob/master/UCI_Datasets/concrete/data/split_data_train_test.py
            n = X.shape[0]
            np.random.seed(1)
            folds = []
            for i in range(5):
                permutation = np.random.choice(range(n), n, replace=False)
                end_train = round(n * 9.0 / 10)
                end_test = n

                train_index = permutation[0:end_train]
                test_index = permutation[end_train:n]
                folds.append((train_index, test_index))        
        else:
            args["lr"] = 0.01
            args["minibatch_frac"] = 1.0 
            kf = KFold(n_splits=args["n_splits"])
            folds = kf.split(X)
            # Follow https://github.com/yaringal/DropoutUncertaintyExps/blob/master/UCI_Datasets/concrete/data/split_data_train_test.py
            n = X.shape[0]
            np.random.seed(1)
            folds = []
            for i in range(args['n_splits']):
                permutation = np.random.choice(range(n), n, replace=False)
                end_train = round(n * 9.0 / 10)
                end_test = n

                train_index = permutation[0:end_train]
                test_index = permutation[end_train:n]
                folds.append((train_index, test_index))


        for itr, (train_index, test_index) in enumerate(folds):
            # print('train_index: ')
            # print(train_index)
            # print('test_index: ')
            # print(test_index)
            X_trainall, X_test = X[train_index], X[test_index]
            y_trainall, y_test = y[train_index], y[test_index]

            X_train, X_val, y_train, y_val = train_test_split(
                X_trainall, y_trainall, test_size=0.2
            )

            y_true += list(y_test.flatten())


            ngb = NGBRegressor(
                Base=base_name_to_learner[args["base"]],
                Dist=eval(args["distn"]),
                Score=eval(args["score"]),
                n_estimators=args["n_est"],
                learning_rate=args["lr"],
                natural_gradient=args["natural"],
                minibatch_frac=args["minibatch_frac"],
                verbose=args["verbose"],
            )

            ngb.fit(X_train, y_train)

            # pick the best iteration on the validation set
            y_preds = ngb.staged_predict(X_val)
            y_forecasts = ngb.staged_pred_dist(X_val)

            val_rmse = [mean_squared_error(y_pred, y_val) for y_pred in y_preds]
            val_nll = [
                -y_forecast.logpdf(y_val.flatten()).mean() for y_forecast in y_forecasts
            ]
            best_itr = np.argmin(val_rmse) + 1

            # re-train using all the data after tuning number of iterations
            ngb = NGBRegressor(
                Base=base_name_to_learner[args["base"]],
                Dist=eval(args["distn"]),
                Score=eval(args["score"]),
                n_estimators=args["n_est"],
                learning_rate=args["lr"],
                natural_gradient=args["natural"],
                minibatch_frac=args["minibatch_frac"],
                verbose=args["verbose"],
            )
            ngb.fit(X_trainall, y_trainall)

            # the final prediction for this fold
            forecast = ngb.pred_dist(X_test, max_iter=best_itr)
            forecast_val = ngb.pred_dist(X_val, max_iter=best_itr)

            # set the appropriate scale if using a homoskedastic Normal
            if args["distn"] == "NormalFixedVar":
                scale = (
                    forecast.var * ((forecast_val.loc - y_val.flatten()) ** 2).mean() ** 0.5
                )
                forecast = norm_dist(loc=forecast.loc, scale=scale)

            ngb_rmse += [np.sqrt(mean_squared_error(forecast.mean(), y_test))]
            ngb_nll += [-forecast.logpdf(y_test.flatten()).mean()]

            print(
                    "[%d/%d] BestIter=%d RMSE: Val=%.4f Test=%.4f NLL: Test=%.4f"
                    % (
                        itr + 1,
                        args['n_splits'],
                        best_itr,
                        np.sqrt(val_rmse[best_itr - 1]),
                        np.sqrt(mean_squared_error(forecast.mean(), y_test)),
                        ngb_nll[-1],
                    )
                )
        # After processing all folds for a dataset:
        end_time = time.time()  # End time measurement
        elapsed_time = end_time - start_time  # Calculate elapsed time
        print(dset)
        print(
                "== GBM=%.4f +/- %.4f, RMSE NGBOOST =%.4f ± %.4f, NLL NGBOOST=%.4f ± %.4f"
                % (
                    0.0,
                    0.0,
                    np.mean(ngb_rmse),
                    np.std(ngb_rmse),
                    np.mean(ngb_nll),
                    np.std(ngb_nll),
                    elsapsed_time
                )
            )
        writer.writerow([
            dset,
            np.mean(ngb_rmse),  # NGBoost RMSE Mean
            np.std(ngb_rmse),   # NGBoost RMSE Std
            np.mean(ngb_nll),   # NGBoost NLL Mean
            np.std(ngb_nll),     # NGBoost NLL Std
            elapsed_time        # Computation Time
        ])

== Dataset=Boston Housing X.shape=(506, 13) LogScore/Normal
[iter 0] loss=3.6606 val_loss=0.0000 scale=1.0000 norm=6.8427
[iter 100] loss=2.7489 val_loss=0.0000 scale=2.0000 norm=5.1619
[iter 200] loss=2.1824 val_loss=0.0000 scale=2.0000 norm=3.4418
[iter 300] loss=1.8918 val_loss=0.0000 scale=2.0000 norm=2.9268
[iter 400] loss=1.7507 val_loss=0.0000 scale=1.0000 norm=1.3560
[iter 500] loss=1.6441 val_loss=0.0000 scale=1.0000 norm=1.2667
[iter 600] loss=1.5586 val_loss=0.0000 scale=1.0000 norm=1.1980
[iter 700] loss=1.4893 val_loss=0.0000 scale=1.0000 norm=1.1456
[iter 800] loss=1.4238 val_loss=0.0000 scale=1.0000 norm=1.0966
[iter 900] loss=1.3550 val_loss=0.0000 scale=1.0000 norm=1.0448
[iter 1000] loss=1.2922 val_loss=0.0000 scale=1.0000 norm=1.0013
[iter 1100] loss=1.2268 val_loss=0.0000 scale=1.0000 norm=0.9604
[iter 1200] loss=1.1625 val_loss=0.0000 scale=1.0000 norm=0.9223
[iter 1300] loss=1.1109 val_loss=0.0000 scale=1.0000 norm=0.8932
[iter 1400] loss=1.0641 val_loss=0.0000 sc

In [72]:
# the final prediction for this fold
forecast = lgblss.predict(X_test)
forecast_val = lgblss.predict(X_val)

lss_rmse += [np.sqrt(mean_squared_error(forecast['loc'].values, y_test))]
val_rmse = [np.sqrt(mean_squared_error(forecast_val['loc'].values, y_val))]
lss_nll += [-norm(forecast['loc'], forecast['scale']).logpdf(y_test.flatten()).mean()]

print(
        "[%d/%d] BestIter=%d RMSE: Val=%.4f Test=%.4f NLL: Test=%.4f"
        % (
            itr + 1,
            args['n_splits'],
            lgblss.booster.best_iteration,
            np.sqrt(val_rmse),
            np.sqrt(mean_squared_error(forecast['loc'].values, y_test)),
            lss_nll[-1],
        )
    )

[1/20] BestIter=0 RMSE: Val=2.5229 Test=7.0629 NLL: Test=3.1941
